# Opinion Survey Analysis

Author: Jennifer Le  
Date: 12/6/24

The purpose of this analysis is to   
1. Examine how opinions on topics like the economy has changed from 2020 to 2024.  
2. Discover groups that are affected the most when opinions about the econmy decline

The data used for this project is from the National Public Opinion Reference surveys (NPORS) conducted by the Pew Reserach Center. 

This survey includes questions on demographics like income, education, and race. It also includes questions about political views, social media use, opinions on the economy, piety and dedication to religious beliefs, and more. Click on the link below to see the 2022 Questionnaire. Navigate to other folders to see questionnaires for other years. 

<a href="https://github.com/JenniferMLe/Opinion-Survey-Analysis/blob/main/Datasets/NPORS-2022/Questionnaire_22.pdf" target="_blank">See questions asked for 2022 study</a>  
<a href="https://www.pewresearch.org/methods/fact-sheet/national-public-opinion-reference-survey-npors/" target="_blank">More information on NPORS</a>  
<a href="https://jennifermle.github.io/Opinion-Survey-Analysis/" 
target="_blank">Click to see graphs (if viewing main.ipynb)</a>

The concept for this project stems from an assignment from Prof. Elena Zheleva's Intro to Data Science class (CS 418), spring 2024.

## Helper Functions

In [168]:
import pandas as pd
import plotly.express as px
from pandasql import sqldf
import numpy as np

def write_to_file(df,file_name='Result_Datasets/result.csv'):
    df.to_csv(file_name, index=False)

# print a list of distinct values from a column in a dataframe
# helps with data cleaning
def get_distinct_values(df, columns, sort_by):
    vals = df[columns].sort_values(by=sort_by).drop_duplicates()
    # print to csv file to see all values since output may be cut if too long
    write_to_file(vals)

def get_count(cols):
    # as_index=False doesn't make group labels the index
    df = df_combined.groupby(cols, as_index=False)['RESPID'].count()
    df = df.rename(columns={'RESPID':'Count'})
    for col in cols:
        df = df[df[col] != 'N/A']
    write_to_file(df)
    return df

category_orders = {
    'AGEGRP2':['18-24','25-39','40-59','60-79','80+'],

    'INCOMEGRP':['< $40K','$40-70K','$70-100K','$100K+'],

    'ECON1MOD':['Poor','Only fair','Good','Excellent'],

    'ECON1BMOD':['Better','About the same','Worse'],
    
    'EDUCATION':[
        "No schooling completed",
        "Some High School",
        "High School",
        "Some College",
        "Associate's Degree",
        "Bachelor's Degree",
        "Master's Degree or Higher"
    ]
}

income_map2 = {
    'Under $40k':"#3d005e",
    '$40-70K':"#8a03d5",
    '$70-100K':"#bd45ff",
    '$100K+':"#d995ff"
}


## Importing Data

In [100]:
# Save each .sav as a data frame
df_20 = pd.read_spss("Datasets/NPORS-2020/dataset.sav")
df_21 = pd.read_spss("Datasets/NPORS-2021/dataset.sav")
df_22 = pd.read_spss("Datasets/NPORS-2022/dataset.sav")
df_23 = pd.read_spss("Datasets/NPORS-2023/dataset.sav")
df_24 = pd.read_spss("Datasets/NPORS-2024/dataset.sav")
df_25 = pd.read_spss("Datasets/NPORS-2025/dataset.sav")

# convert .sav to .csv
write_to_file(df_20,"Datasets/NPORS-2020/dataset_20.csv")
write_to_file(df_21,"Datasets/NPORS-2021/dataset_21.csv")
write_to_file(df_22,"Datasets/NPORS-2022/dataset_22.csv")
write_to_file(df_23,"Datasets/NPORS-2023/dataset_23.csv")
write_to_file(df_24,"Datasets/NPORS-2024/dataset_24.csv")
write_to_file(df_25,"Datasets/NPORS-2025/dataset_25.csv")

## Combining Data

In [101]:
# stop number of columns at the end that don't have the year in the column name
def remove_year_from_column_name(dataset, stop):
    list_columns = list(dataset.columns)

    for i in range(0,len(list_columns)-stop):
        list_columns[i] = list_columns[i][:-5]

    dataset.columns = list_columns

# Remove the year from column name to keep naming consistant
remove_year_from_column_name(df_20, 5)
remove_year_from_column_name(df_21, 2)

# ensure columns recording the same data have the same name
# so there aren't duplicate columns when appending datasets
df_20 = df_20.rename(columns={
    'SEXASK':'GENDER', 
    'EDUC_ACS':'EDUCATION'
})

df_23 = df_23.rename(columns={
    'BASEWT':'BASEWEIGHT', 
    'INC_SDT1':'INCOME'
})

df_24 = df_24.rename(columns={
    'BASEWT':'BASEWEIGHT', 
    'INC_SDT1': 'INCOME',
    'SMUSEa' : 'SMUSE_a','SMUSEd' : 'SMUSE_d','SMUSEg' : 'SMUSE_g','SMUSEj' : 'SMUSE_j',
    'SMUSEb' : 'SMUSE_b','SMUSEe' : 'SMUSE_e','SMUSEh' : 'SMUSE_h','SMUSEk' : 'SMUSE_k',
    'SMUSEc' : 'SMUSE_c','SMUSEa' : 'SMUSE_f','SMUSEi' : 'SMUSE_i'
})

df_25 = df_25.rename(columns={
    'BASEWT':'BASEWEIGHT', 
    'INC_SDT1': 'INCOME'
})

# add year columns to all datasets
df_20['YEAR'] = 2020
df_21['YEAR'] = 2021
df_22['YEAR'] = 2022
df_23['YEAR'] = 2023
df_24['YEAR'] = 2024
df_25['YEAR'] = 2025

# Combine (append) all datasets together
df_combined = pd.concat([df_20, df_21, df_22, df_23, df_24, df_25])
print(df_combined.shape)

# Keep relevent columns only
df_combined = df_combined[[
    'RESPID','YEAR','AGE','AGEGRP','GENDER','RACECMB', # basic demographics
    'INCOME','EDUCATION', 'RELIG', 'PARTY', # other useful demographics 
    'RELIMP', 'PRAY', 'MARITAL', # other features
    'SMUSE_a', 'SMUSE_b', 'SMUSE_c', 'SMUSE_d', 'SMUSE_e', 'SMUSE_f', # social media use
    'SMUSE_g','SMUSE_h','SMUSE_i','SMUSE_j','SMUSE_k',
    'ECON1MOD', 'ECON1BMOD', # we want to study how this changes over time
    'BASEWEIGHT', 'WEIGHT' 
]]


(28469, 123)


## Cleaning Data

In [102]:
# rename columns
df_combined = df_combined.rename(columns={
    'SMUSE_a':'FACEBOOK',
    'SMUSE_b':'YOUTUBE',
    'SMUSE_c':'TWITTER',
    'SMUSE_d':'INSTAGRAM',
    'SMUSE_e':'SNAPCHAT',
    'SMUSE_f':'WHATSAPP',
    'SMUSE_g':'LINKEDIN',
    'SMUSE_h':'PINTEREST',
    'SMUSE_i':'TIKTOK',
    'SMUSE_j':'BEREAL',
    'SMUSE_k':'REDDIT',
    'RACECMB':'RACE'
})

'''
replace values for consistency 
'''
# change n/a to -1 so we can convert age to float
df_combined["AGE"] = df_combined["AGE"].replace({
    "n/a":"-1",
    "98+":"98",
    "":"-1",
    'Refused':"-1"
})

# change column type
df_combined["AGE"] = df_combined["AGE"].astype(float)


df_combined = df_combined.replace({
    r'.*Refused.*':'N/A',
    r'.*Something else.*':'Other',
    'No, don\'t use this':'No',
    'No, don’t use this':'No',
    "Yes, use this":'Yes',
},regex=True)

df_combined['GENDER'] = df_combined['GENDER'].replace({
    "A man":"Male",
    "A woman":"Female",
    "In some other way":"Other"
})

df_combined["RACE"] = df_combined["RACE"].replace({
    r'.*Asian.*':'Asian',
    r'.*Black.*':'Black',
    r'.*other.*':'Other',
    'Mixed race':'Mixed Race'
},regex=True)

df_combined['ECON1MOD'] = df_combined['ECON1MOD'].replace('Only Fair','Only fair')

df_combined["INCOME"] = df_combined["INCOME"].replace({
    r' to less than ':'-',
    r' or more':'+',
    r'Less than':'<',
    r',000':'K',
},regex=True)

df_combined["EDUCATION"] = df_combined["EDUCATION"].replace({
    r'.*11.*':'Some High School',
    r'.*12.*':'Some High School',
    r'.*high school.*':'High School',
    r'.*GED.*':'High School',
    r'.*college.*':'Some College',
    r'.*Associate.*':'Associate\'s Degree',
    r'.*Bachelor.*':'Bachelor\'s Degree',
    r'.*Master.*':'Master\'s Degree or Higher',
    r'.*MD.*':'Master\'s Degree or Higher',
    r'.*Doctorate.*':'Master\'s Degree or Higher'
},regex=True)

df_combined["RELIG"] = df_combined["RELIG"].replace({
    r'.*Mormon.*':'Mormon',
    r'.*Orthodox.*':'Orthodox',
    r'.*Protestant.*':'Protestant'
},regex=True)

df_combined['MARITAL'] = df_combined['MARITAL'].replace({
    "Separated":"Divorced",
    "Never been married":"Never married",
})

categorical_cols = df_combined.select_dtypes(include="category").columns
df_combined[categorical_cols] = df_combined[categorical_cols].astype(str)

write_to_file(df_combined, 'Result_Datasets/combined_dataset.csv')
print(df_combined.shape)


(28469, 28)


## Creating Calculated Columns

In [103]:
'''create calculated columns to group incomes'''
# conditions for each group
conditions = [
    df_combined['INCOME'].isin(['$10K-$20K','$20K-$30K','$30K-$40K','< $10K','< $30K']),
    df_combined['INCOME'].isin(['$40K-$50K','$50K-$60K','$60K-$70K','$50K-$70K']),
    df_combined['INCOME'].isin(['$70K-$100K','$70K-$80K','$70K-$90K','$75K-$100K','$80K-$90K','$90K-$100K']),
    df_combined['INCOME'].isin(['$100K+','$100K-$125K','$100K-$150K','$125K-$150K','$150K+'])
]
# corresponding groups for each condition
group = ['< $40K','$40-70K','$70-100K','$100K+']

# insert new column after INCOME
df_combined.insert(
    df_combined.columns.get_loc('INCOME') + 1, # position we want to insert at
    'INCOMEGRP', # name of new column
    np.select(conditions, group, default='N/A') # set value according to conditions 
)

'''create calculated columns to group incomes'''
conditions = [
    (df_combined['AGEGRP'] == '18-24') | ((18 <= df_combined['AGE']) & (df_combined['AGE'] <= 24)),
    (df_combined['AGEGRP'].isin(['25-29','30-34','35-39'])) | ((25 <= df_combined['AGE']) & (df_combined['AGE'] <= 39)),
    (df_combined['AGEGRP'].isin(['40-44','45-49','50-54','55-59'])) | ((40 <= df_combined['AGE']) & (df_combined['AGE'] <= 59)),
    (df_combined['AGEGRP'].isin(['60-64','65-69','70-74','75-79'])) | ((60 <= df_combined['AGE']) & (df_combined['AGE'] <= 79)),
    (df_combined['AGEGRP'] == '80+')| (80 <= df_combined['AGE'])
]
group = ['18-24','25-39','40-59','60-79','80+']

df_combined.insert(
    df_combined.columns.get_loc('AGEGRP') + 1, # position we want to insert at
    'AGEGRP2', # name of new column
    np.select(conditions, group, default='N/A') # set value according to conditions 
)

In [115]:
df = df_combined[['AGE','AGEGRP','AGEGRP2']].sort_values(by='AGE').drop_duplicates()


write_to_file(df_combined.iloc[0:10])

## Suvery Participant Demographics

In this section, we will get to know our 2020-2024 survey participants and who they consist of by examining their age, race, gender, income, and education.

In [85]:
'''Age Distribution'''

fig_age = px.bar(
    get_count(['AGEGRP2']), 
    x = 'AGEGRP2', 
    y = 'Count',
    title = "Demographic Breakdown by Age",
    category_orders = category_orders,
    color_discrete_sequence=px.colors.sequential.Viridis
)
fig_age.update_xaxes(title="Age").show()

'''Race Distribution'''

fig_race = px.bar(
    get_count(['RACE']), 
    y='RACE', 
    x='Count',
    title='Demographic Breakdown by Race',
    text_auto='.2s', # put numbers in K format
    color_discrete_sequence=['#ffd525']
    
)
# add subtitle
fig_race.update_layout(
    annotations=[
        dict(
            x=-0.45, y=-0.20,  # Position below the chart
            text="*Participants can select more than 1 race",
            showarrow=False,  # No arrow pointing to the text
            xref="paper", yref="paper",  # Position relative to the chart
            font=dict(size=12, color="gray"),
        )
    ]
)
fig_race.update_yaxes(title="").update_layout(yaxis={'categoryorder':'total descending'}).show()

'''Income Distribution'''

fig_income = px.bar(
    get_count(['INCOMEGRP']),
    x='INCOMEGRP',
    y='Count',
    title='Demographic Breakdown by Income Group',
    text_auto='.2s',
    color_discrete_sequence=['#77be34'],
    category_orders= category_orders
)

# add subtitle
fig_income.update_layout(
    annotations=[
        dict(
            x=-0.08, y=-0.20,  # Position below the chart
            text="*Participants in the $50-75K group (665 people) are excluded",
            showarrow=False,  # No arrow pointing to the text
            xref="paper", yref="paper",  # Position relative to the chart
            font=dict(size=12, color="gray"),
        )
    ]
).update_xaxes(title="Income Group").show()

'''Education Distribution'''

fig_education = px.bar(
    get_count(['EDUCATION']),
    y = 'EDUCATION',
    x = 'Count',
    title = 'Demographic Breakdown by Education',
    color_discrete_sequence=['#063dcf'], 
    text='Count',
    category_orders=category_orders
)
fig_education.update_yaxes(title="Education").show()


## Exploring Changes in Opinions from 2020 to 2024

In [49]:
color_map = {
    'Excellent':"#0daa00",
    'Good':'#8fdc32',
    'Only fair':'#ffc500',
    'Poor':'#f6492a',
    'Better':"#8fdc32",
    'About the same':'#ffc500',
    'Worse':'#f6492a'
}

In [111]:
# this function displays a cluster bar graph dispalying yearly changes in a column in the combined dataframe
def examine_changes(column, title):
    df_change = get_count(['YEAR',column])
    df_change['Percent'] = (df_change['Count'] / df_change.groupby('YEAR')['Count'].transform('sum')) * 100
    write_to_file(df_change)

    fig = px.line(
        df_change,
        x='YEAR',
        y='Percent',
        color=column,
        # barmode='group',
        title=title,
        hover_data=['Count'],
        category_orders=category_orders,
        color_discrete_map=color_map
    )
    # type='category' force x-axis to be discrete categories, not continuous bins so years won't be combined
    fig.update_yaxes(title="Percentage").update_xaxes(type='category').update_traces(line=dict(width=8)).show()

In [112]:
examine_changes('ECON1MOD','Yearly Breakdown by how Particapates Rate their Community\'s Economic Conditions')

examine_changes('ECON1BMOD','Yearly Breakdown by Opinions about the Economic Conditions of their Community in 1 Year')

Opinions on the current economic conditions and estimated economic conditions in a year became more negative from 2021 to 2022. 

The percent of survey participants who believed the economy was going to get worse in a year increased from 21% to 37% from 2021 to 2022. The percent of survey participants who believed their community's economy was poor increased from 10% to 21%.

This could be caused by record-breaking inflation rates that occured towards the end of 2021 and throughout 2022. 

Source: [United States Inflation Rate](https://tradingeconomics.com/united-states/inflation-cpi)   

## Going Into the Details

Next let's explore which groups are causing this increase in negative sentiment about the economy, especially in 2022. 

In [99]:
get_distinct_values(df_combined,['INCOME'],'INCOME')

In [227]:
'''
Create a df with these columns
1. percent difference of ECON1MOD = good and excellent from 2021 to 2022
2. percent different of ECON1MOD = only fair and poor from 2021 to 2022

Each row represents a group, e.g for income, one row represents <$40K

What we need

group e.g <40K
year e.g 2021
percent of group that answered positively in 2021
percent of group that answer positively in 2022
percent diff for positive answers

percent of group tha answered negatively in 2021
percent of group that answered negatively in 2022
percent diff for negative answers

Group by income group, econ1mod, 
'''

def graph_percent_increase(df):
    fig = px.bar(
        df,
        x='YEAR',
        y='value',
        barmode='group',
        color=df.columns[0],
        hover_data=['YEAR'],
        category_orders=category_orders,
        color_discrete_map=color_map
    )
    # type='category' force x-axis to be discrete categories, not continuous bins so years won't be combined
    fig.update_yaxes(title="Percentage").update_xaxes(type='category').show()

def get_percent_increase(column):
    df = get_count(['YEAR',column,'ECON1MOD'])

    conditions = [
        (df['ECON1MOD'] == 'Excellent') | (df['ECON1MOD'] == 'Good'),
        (df['ECON1MOD'] == 'Poor') | (df['ECON1MOD'] == 'Only fair')
    ]
    group = ['Positive', 'Negative']

    df['SENTIMENT'] = np.select(conditions, group, default='N/A')

    # df = df[
    #     (df['YEAR'] == 2021) | (df['YEAR'] == 2022)
    # ]

    df = df.groupby(['YEAR',column,'SENTIMENT'],as_index=False)['Count'].sum()
    df = pd.pivot_table(df,index=['YEAR',column],columns=['SENTIMENT']).reset_index()

    new_columns = []
    for col in df.columns:
        new_name = ''
        for x in col:
            if isinstance(x, str):
                new_name += x
            else:
                new_name += str(x)
        new_columns.append(new_name)
    df.columns = new_columns

    df['PercentNeg'] = round((df['CountNegative'] / (df['CountNegative']+df['CountPositive']))*100,1)

    df = pd.pivot_table(df,index=column,columns='YEAR',values='PercentNeg').reset_index()
    
    for i in range(2021,2026):
        col_name = 'diff' + str(i)
        df[col_name] = round(df[i] - df[i-1],1)
    
    
    df = pd.melt(df,id_vars = ['INCOMEGRP'], value_vars=['diff2021','diff2022','diff2023','diff2024','diff2025'])
    write_to_file(df)
    return df

get_percent_increase('INCOMEGRP')

graph_percent_increase(get_percent_increase('INCOMEGRP'))
